# Veezha Veera — Full-Song Layer 3 Remix (Colab T4 GPU)

Runs the **full 238-second** song through the Autonomous Remix Studio's `main.py`
on a Colab T4 GPU: real BS-RoFormer stem separation + real MusicGen-Melody
beat / melody / riser + DAW export (REAPER `.RPP` + `stems_and_midi/`).

**Before you press Run all:**

1. `Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save`
2. `Runtime → Run all`

**What to expect:** setup ~10–15 min (deps + model downloads), remix run
~20–40 min on T4 (vs ~6 hours on CPU). All outputs land on Google Drive at
`MyDrive/remix-studio-outputs/veezha-full-l3/`, so they survive even if the
Colab session disconnects.

**Free-tier notes:** a Colab session lasts up to 12 h and goes idle after
~90 min without activity — keep the tab open while the remix cell runs. If
it disconnects, just `Runtime → Run all` again from the top. The models
download into the Colab VM (not Drive), so a reconnect re-downloads them
(~5–10 min); outputs already on Drive are never lost.

**Private use only:** "Veezha Veera" is a copyrighted song (Sai Abhyankkar,
film OM / Dhanush, Divo Music). This notebook, the remix and its outputs are
for private listening only — never publish, share or sell them.

In [ ]:
# 1 — GPU check (nvidia-smi only; torch is installed in the next steps,
# so we deliberately do NOT import torch here yet).
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2 — Mount Google Drive (code zip + input wav live there).
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
# 3 — Stage the project code + input wav from Drive.
import shutil
import zipfile
from pathlib import Path

CODE_ZIP = Path('/content/drive/MyDrive/remix-studio/studio-code.zip')
WAV_SRC = Path('/content/drive/MyDrive/remix-studio/veezha-veera-44100.wav')
PROJ = Path('/content/remix-studio')

assert CODE_ZIP.is_file(), f'Code zip not found on Drive: {CODE_ZIP}'
assert WAV_SRC.is_file(), f'Input wav not found on Drive: {WAV_SRC}'

if PROJ.exists():
    shutil.rmtree(PROJ)
PROJ.mkdir(parents=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(PROJ)

# Input wav -> local disk (faster than reading it off the Drive mount).
inputs = PROJ / 'inputs'
inputs.mkdir(exist_ok=True)
WAV = inputs / 'veezha-veera-44100.wav'
shutil.copy2(WAV_SRC, WAV)
print(f'staged project: {PROJ}')
print(f'input wav: {WAV} ({WAV.stat().st_size // 1024 // 1024} MB)')
print('torchaudio shim dist-info present:',
      (PROJ / 'models/_shims/torchaudio-0.0.0.dist-info/METADATA').is_file())

In [ ]:
# 4 — Colab-only GPU patch (applies to THIS copy in /content — the project
# source on disk / Drive is never touched).
#
# The stem server hardcodes torch_device='cpu' for BS-RoFormer (it was
# written on a CPU-only box). On Colab we point it at CUDA; everything
# else in the codebase already picks up the GPU via core.config.get_device().
# Autocast stays OFF (fp32 on T4 is fast enough and safest for the stems).
p = PROJ / 'mcp_servers/stem_theory_server.py'
src = p.read_text()
old = '"torch_device": torch.device("cpu"),'
new = '"torch_device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),'
assert old in src, 'Patch pattern not found — stem_theory_server.py changed upstream!'
assert new not in src, 'Patch already applied?'
p.write_text(src.replace(old, new, 1))
print('patched: stem separator will run on CUDA (Colab copy only)')
print('verify:', [l.strip() for l in p.read_text().splitlines() if 'torch_device":' in l][:1])

## Install notes — what differs from the project's pins, and why

- **torch 2.15.0 from the CUDA index (`cu128`)** — same version as the
  project pin, but the CUDA wheel instead of the CPU wheel. Install torch
  FIRST, otherwise pip resolves the CPU wheel from default PyPI. (If the
  cu128 install fails on this runtime, retry with `cu126`.)
- **transformers 4.57.1, audio-separator 0.48.0, audioread 3.1.0,
  huggingface_hub 0.36.2, PyYAML 6.0.3** — installed in the project's venv
  but NOT listed in `requirements.txt`; pinned here to the venv's versions.
- **Skipped: essentia, pyrubberband, fastapi, uvicorn** — nothing in the
  codebase imports them (`pyrubberband` is a lazy optional import with a
  librosa fallback; the `rubberband` CLI binary isn't available on Colab).
- **torchvision / torchaudio: not installed explicitly.** torchvision 0.29.2
  (the torch 2.15-era build) has broken C++ ops and torchaudio has no 2.15
  build — the generative server already handles both with in-process shims,
  and the code zip carries the `models/_shims/` dist-info the torchaudio
  stub needs. If pip pulls torchvision in as a dependency, the same shims
  cover it.
- **MCP timeouts already 3600 s** in `main.py` (raised for the CPU run) —
  kept as-is; on T4 the beat is far faster, so this is pure headroom.

In [ ]:
# 5 — Install deps: torch FIRST from the CUDA index, then the rest.
!pip install -q torch==2.15.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q langgraph==1.2.22 langchain-core==1.8.1 fastmcp==4.2.0 pydantic==2.14.3 librosa==1.0.1 pedalboard==0.9.26 matchering==2.0.6 pretty_midi==0.2.11.post0 soundfile==0.14.0 numpy==2.5.5 scipy==1.18.1 python-dotenv==1.2.4 transformers==4.57.1 audio-separator==0.48.0 audioread==3.1.0 huggingface_hub==0.36.2 PyYAML==6.0.3
print('pip installs done')

In [ ]:
# 6 — Verify the installs (torch must see the GPU now).
import importlib.metadata as md
import torch

print('torch:', torch.__version__, '| CUDA build:', torch.version.cuda,
      '| available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'torch cannot see the GPU — runtime type must be T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
for pkg in ['transformers', 'fastmcp', 'librosa', 'audio-separator',
            'pedalboard', 'soundfile', 'numpy', 'scipy']:
    try:
        print(f'{pkg}: {md.version(pkg)}')
    except Exception as e:
        print(f'{pkg}: MISSING ({e})')
import librosa, audio_separator, pedalboard, soundfile  # noqa: F401 — import check
print('install check OK ✅')

In [ ]:
# 7 — Download both models from Hugging Face (public repos, no token)
# into the exact paths the code expects.
from huggingface_hub import snapshot_download

snapshot_download(repo_id='SYH99999/bs_roformer_4stems_ft',
                  local_dir=str(PROJ / 'models/bs_roformer_4stems_ft'))
snapshot_download(repo_id='facebook/musicgen-melody',
                  local_dir=str(PROJ / 'models/musicgen-melody'))

# Verify the exact files the servers require.
assert (PROJ / 'models/bs_roformer_4stems_ft/bs_roformer_4stems_ft.pth').is_file(), 'RoFormer weights missing!'
assert (PROJ / 'models/bs_roformer_4stems_ft/config.yaml').is_file(), 'RoFormer config missing!'
assert list((PROJ / 'models/musicgen-melody').glob('*.safetensors')), 'MusicGen safetensors missing!'
print('models ready ✅')
print('  bs_roformer_4stems_ft: %.0f MB' % ((PROJ / 'models/bs_roformer_4stems_ft/bs_roformer_4stems_ft.pth').stat().st_size / 1e6))
print('  musicgen-melody files:', len(list((PROJ / 'models/musicgen-melody').iterdir())))

In [ ]:
# 8 — Smoke check: the project must import and the CLI must parse
# (main.py imports the agents + graph at module load).
%cd /content/remix-studio
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!python main.py --help

In [ ]:
# 9 — RUN: full 238 s song, real models, DAW export. Output + log go to
# Drive so nothing is lost if the session disconnects. ~20–40 min on T4.
import os
import subprocess
import sys
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
PROJ = Path('/content/remix-studio')
WAV = PROJ / 'inputs/veezha-veera-44100.wav'
assert WAV.is_file(), f'Input wav missing: {WAV} — re-run the staging cell'
OUT = Path('/content/drive/MyDrive/remix-studio-outputs/veezha-full-l3')
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / 'run.log'

cmd = [sys.executable, '-u', 'main.py',
       '--input', str(WAV),
       '--output', str(OUT),
       '--export-daw']
print('running:', ' '.join(cmd))
print('log:', LOG)

proc = subprocess.Popen(cmd, cwd='/content/remix-studio',
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
with open(LOG, 'w') as f:
    for line in proc.stdout:
        f.write(line)
        f.flush()
        print(line, end='', flush=True)
rc = proc.wait()
print(f'\nexit code: {rc}')
assert rc == 0, f'remix exited {rc} — see {LOG}'
print('remix finished ✅')

In [ ]:
# 10 — Verify the outputs on Drive.
import soundfile as sf
from pathlib import Path

OUT = Path('/content/drive/MyDrive/remix-studio-outputs/veezha-full-l3')
print('== output files ==')
for f in sorted(OUT.iterdir()):
    print(f'  {f.name}  {f.stat().st_size / 1e6:.1f} MB')

master = OUT / 'final_master_full.wav'
info = sf.info(str(master))
dur = info.frames / info.samplerate
print(f'\nmaster: {info.samplerate} Hz, {info.channels} ch, {info.subtype}, {dur:.1f} s')
assert info.samplerate == 44100 and info.channels == 2 and info.subtype == 'PCM_24', 'master format wrong!'
assert abs(dur - 238) < 2.0, f'master length {dur:.1f} s != ~238 s'

hook = OUT / 'viral_hook_30s.wav'
hinfo = sf.info(str(hook))
print(f'hook: {hinfo.frames / hinfo.samplerate:.1f} s, {hinfo.subtype}')
print('remix.RPP:', (OUT / 'remix.RPP').is_file())
sm = OUT / 'stems_and_midi'
print('stems_and_midi:', sorted(p.name for p in sm.iterdir()) if sm.is_dir() else 'MISSING')

print('\n== verdict (from run.log) ==')
log = (OUT / 'run.log').read_text()
for line in log.splitlines():
    if any(k in line for k in ('master:', 'hook:', 'approved:', 'lufs:', 'daw_project:')):
        print(' ', line)
print('\n== fallback scan (any line mentioning fallback = DSP stood in for a model) ==')
fb = [l for l in log.splitlines() if 'fallback' in l.lower()]
print('\n'.join('  ' + l for l in fb) if fb else '  none — real models ran throughout ✅')